In [13]:

from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

set_llm_cache(SQLiteCache(database_path="cache/langchain.db"))

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

token_provider = get_bearer_token_provider(  
            DefaultAzureCredential(),  
            "https://cognitiveservices.azure.com/.default"  
        )

In [14]:
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI

model = AzureChatOpenAI(
    deployment_name="gpt-4o",
    azure_ad_token_provider = token_provider,
    temperature = 0
    )

embeddings = AzureOpenAIEmbeddings(
    azure_deployment = 'text-embedding-ada-002',
    azure_ad_token_provider = token_provider
)

In [15]:
import pandas as pd

feeds = pd.read_excel('data_excel/feeds.xlsx')
feeds

,link,guid,type,id,sponsored,title,description,pubDate
0,https://www.cnbc.com/2025/06/25/trump-iran-int...,108163859,cnbcnewsstory,108163859,False,Trump fumes over U.S. intelligence report on I...,Trump ordered U.S. strikes on Iranian nuclear ...,"Wed, 25 Jun 2025 15:40:29 GMT"
1,https://www.cnbc.com/2025/06/25/nvidia-shares-...,108163925,cnbcnewsstory,108163925,False,Nvidia shares head for record close as Wall St...,Nvidia matched its intraday high and headed fo...,"Wed, 25 Jun 2025 16:24:21 GMT"
2,https://www.cnbc.com/2025/06/25/may-2025-new-h...,108163831,cnbcnewsstory,108163831,False,"Sales of new homes tanked in May, pushing supp...",New home sales dropped much more than expected...,"Wed, 25 Jun 2025 17:01:11 GMT"
3,https://www.cnbc.com/2025/06/24/stock-market-t...,108163476,live_story,108163476,False,S&P 500 is little changed as index approaches ...,The S&P 500 hovered near the flatline on Wedne...,"Wed, 25 Jun 2025 17:16:22 GMT"
4,https://www.cnbc.com/2025/06/25/why-blackrocks...,108163895,cnbcnewsstory,108163895,False,Why BlackRock's Rick Rieder is confident in eq...,BlackRock's Rick Rieder is confident a stock m...,"Wed, 25 Jun 2025 17:06:37 GMT"
...,...,...,...,...,...,...,...,...
145,https://www.cnbc.com/2025/06/18/cnbcs-uk-excha...,108160309,cnbcnewsstory,108160309,False,CNBC's UK Exchange newsletter: A quantum quand...,The government's supercomputer U-turn is welco...,"Wed, 18 Jun 2025 05:30:01 GMT"
146,https://www.cnbc.com/2025/06/17/spotifys-danie...,108160256,cnbcnewsstory,108160256,False,Spotify's Daniel Ek leads $694 million investm...,Defense tech has become a hot area for investo...,"Tue, 17 Jun 2025 12:41:59 GMT"
147,https://www.cnbc.com/2025/06/17/shell-totalene...,108160271,cnbcnewsstory,108160271,False,Top oil CEOs sound the alarm as Israel-Iran st...,Some energy facilities have been hit in both I...,"Tue, 17 Jun 2025 15:18:11 GMT"
148,https://www.cnbc.com/2025/06/17/gold-outshines...,108160140,cnbcnewsstory,108160140,False,"Gold outshines Treasurys, yen and Swiss franc ...",\Gold's key advantage is that it is no one els...,"Tue, 17 Jun 2025 05:08:01 GMT"


In [16]:
feeds_desc = feeds['title'] + feeds['description']
feeds_desc

0      Trump fumes over U.S. intelligence report on I...
1      Nvidia shares head for record close as Wall St...
2      Sales of new homes tanked in May, pushing supp...
3      S&P 500 is little changed as index approaches ...
4      Why BlackRock's Rick Rieder is confident in eq...
                             ...                        
145    CNBC's UK Exchange newsletter: A quantum quand...
146    Spotify's Daniel Ek leads $694 million investm...
147    Top oil CEOs sound the alarm as Israel-Iran st...
148    Gold outshines Treasurys, yen and Swiss franc ...
149    Stands shuttered at some Israeli defense firm ...
Length: 150, dtype: object

In [17]:
from langchain_core.documents import Document

documents = []

for desc in feeds_desc:
    doc = Document(
        page_content=str(desc)
        )
    documents.append(doc)

documents = documents[:50]
documents

In [18]:
batch_inputs = []
for doc in documents:
    batch_inputs.append({"full_text": doc.page_content})
    
batch_inputs

[Document(metadata={}, page_content='Trump fumes over U.S. intelligence report on Iran strikes at NATO presser Trump ordered U.S. strikes on Iranian nuclear sites as part of its support for Israel, then days later announced a ceasefire agreement between the countries.'),
 Document(metadata={}, page_content='Nvidia shares head for record close as Wall Street shrugs off China concernsNvidia matched its intraday high and headed for a record close as investors gain confidence that the chipmaker will power through China restrictions.'),
 Document(metadata={}, page_content='Sales of new homes tanked in May, pushing supply up to a 3-year highNew home sales dropped much more than expected, as high prices, high mortgage rates and weak consumer confidence ate into demand. '),
 Document(metadata={}, page_content='S&P 500 is little changed as index approaches record high: Live updatesThe S&P 500 hovered near the flatline on Wednesday as investors watched to see if the benchmark index could return 

In [19]:
# from langchain_community.vectorstores.faiss import FAISS

# vs = FAISS.from_documents(documents, embeddings)

In [32]:
from langchain import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = """You are a summary writer. Summarize the following text in about 50 words.: {full_text}"""
prompt = PromptTemplate(
        template=template,
    input_variables=['full_text']
)
chain = prompt | model | StrOutputParser()
chain.batch(batch_inputs[:5])


In [37]:
def summarize(model, batch_inputs):
    template = """You are a summary writer. Summarize the following text in about 50 words.: {full_text}"""
    prompt = PromptTemplate(
            template=template,
        input_variables=['full_text']
    )
    chain = prompt | model | StrOutputParser()
    return chain.batch(batch_inputs[:5])

In [38]:
summarize(model, batch_inputs)

['At a NATO press conference, Trump expressed anger over a U.S. intelligence report concerning Iran. Initially, he ordered strikes on Iranian nuclear sites to support Israel, but subsequently announced a ceasefire agreement between the nations, indicating a shift in his approach to the conflict.',
 "Nvidia's shares are approaching a record close, with investors optimistic despite concerns about China restrictions. The chipmaker's stock matched its intraday high, reflecting confidence in its ability to navigate challenges and continue its growth trajectory.",
 'In May, new home sales significantly declined due to high prices, elevated mortgage rates, and weak consumer confidence, leading to a surge in supply reaching a three-year high.',
 'On Wednesday, the S&P 500 remained nearly unchanged as investors closely monitored its movement, anticipating whether the benchmark index could reach its record high.',
 "BlackRock's Rick Rieder is optimistic about equities in the second half of the y

In [69]:
def trading_view(model, batch_inputs):
    template = """
        You are an expert trader specializing in niche safe-haven assets. 
        Your mission is to generate consistent profits by identifying and capitalizing on opportunities in assets that preserve value during times of global uncertainty. \n
        You possess deep expertise in macroeconomics, commodities markets, liquidity risk management, and the energy sector. \n
        You maintain up-to-the-minute awareness of geopolitical developments, especially escalating conflicts and tensions in the Middle East and the Russia-Ukraine region.\n
        Write a trading recomendation from the following text: {full_text}.\n
        The trading recomendation should contains the following section: \n
        - Asset name\n
        - Current Situation\n
        - Macro Environment\n
        - Geopolitical Considerations\n
        - Market Sentiment\n
        - Risk Management\n
        - Recommendation\n
        
        """
    prompt = PromptTemplate(
            template=template,
        input_variables=['full_text']
    )
    chain = prompt | model | StrOutputParser()
    return chain.batch(batch_inputs)

In [50]:
batch_inputs[:5]

[{'full_text': 'Trump fumes over U.S. intelligence report on Iran strikes at NATO presser Trump ordered U.S. strikes on Iranian nuclear sites as part of its support for Israel, then days later announced a ceasefire agreement between the countries.'},
 {'full_text': 'Nvidia shares head for record close as Wall Street shrugs off China concernsNvidia matched its intraday high and headed for a record close as investors gain confidence that the chipmaker will power through China restrictions.'},
 {'full_text': 'Sales of new homes tanked in May, pushing supply up to a 3-year highNew home sales dropped much more than expected, as high prices, high mortgage rates and weak consumer confidence ate into demand. '},
 {'full_text': 'S&P 500 is little changed as index approaches record high: Live updatesThe S&P 500 hovered near the flatline on Wednesday as investors watched to see if the benchmark index could return to its all-time high. '},
 {'full_text': "Why BlackRock's Rick Rieder is confident i

In [70]:
trading_rec = trading_view(model, batch_inputs)
trading_rec

['**Trading Recommendation**\n\n**Asset Name:** Gold\n\n**Current Situation:**  \nGold is traditionally viewed as a safe-haven asset, and its price tends to rise during periods of geopolitical tension and uncertainty. The recent developments involving U.S. strikes on Iranian nuclear sites and subsequent ceasefire announcements have created a volatile environment, which could lead to increased demand for gold as investors seek stability.\n\n**Macro Environment:**  \nThe global economy is currently facing multiple challenges, including inflationary pressures, supply chain disruptions, and potential interest rate hikes by central banks. These factors contribute to an uncertain macroeconomic environment, making safe-haven assets like gold more attractive to investors looking to preserve value.\n\n**Geopolitical Considerations:**  \nThe Middle East remains a region of significant geopolitical tension, particularly with the recent U.S. actions against Iran and the ongoing support for Israel.

In [72]:
print(trading_rec[1])

**Trading Recommendation**

**Asset Name:** Nvidia Corporation (NVDA)

**Current Situation:**  
Nvidia shares are approaching a record close, driven by investor confidence in the company's ability to navigate through recent China restrictions. The stock has matched its intraday high, indicating strong market interest and momentum.

**Macro Environment:**  
The global economy is experiencing mixed signals, with some regions showing signs of recovery while others face challenges such as inflationary pressures and supply chain disruptions. The technology sector, particularly semiconductor companies like Nvidia, remains crucial due to the increasing demand for chips in various industries, including AI, automotive, and data centers.

**Geopolitical Considerations:**  
Tensions between the U.S. and China have escalated, with restrictions impacting technology companies. However, Nvidia's strategic positioning and diversified customer base may mitigate some of these risks. Additionally, ongoin

In [73]:
print(trading_rec[10])

**Trading Recommendation**

**Asset Name:** Real Estate Investment Trusts (REITs) focusing on data centers

**Current Situation:** Meta Platforms, Inc. is investing $10 billion to build the largest data center in the Western Hemisphere in rural Northeastern Louisiana. This significant investment highlights the growing demand for data infrastructure, driven by increasing digitalization and cloud computing needs.

**Macro Environment:** The global economy is experiencing a shift towards digital transformation, with businesses and consumers relying more heavily on cloud services, data storage, and digital communication. This trend is expected to continue, supporting the growth of data centers as critical infrastructure. Additionally, interest rates remain relatively low, making financing for large-scale infrastructure projects more accessible.

**Geopolitical Considerations:** The geopolitical landscape is marked by tensions in the Middle East and the Russia-Ukraine region, which can lead

In [77]:
from pydantic import BaseModel, Field

from langchain_core.output_parsers import PydanticToolsParser

In [78]:
def trading_view(model, batch_inputs):

    class trading_recomendation(BaseModel):
        asset_name: str = Field(..., description="The name of the assest to invest in.")
        current_situation: str = Field(..., description="The current situation of the asset or of the company.")
        macro_enviroment: str = Field(..., description="The description of the macro enviroment the company in operating in.")
        geopolitical: str = Field(..., description="The geopolitical landscape of the company or of the asset or of the market it belong.")
        sentiment: str = Field(..., description="The sentiment towards the asset.")
        risk_management: str = Field(..., description="The risk involved.")
        Recommendation: str = Field(..., description="Trading Recommendation.")



    prompt_template = """
        Extract a trading recomendation from the following text.
        Preserve the original language of the text.
        Format the output as JSON matching the QAItem schema.
        text: {full_text}
        
        """
    parser = PydanticToolsParser(tools=[trading_recomendation])
    model = model.bind_tools([trading_recomendation])
    prompt = PromptTemplate.from_template(prompt_template)
    chain = prompt | model | parser

    return chain.batch(batch_inputs)

In [79]:
trading_rec2 = trading_view(model, batch_inputs)
trading_rec2

[[trading_recomendation(asset_name='U.S. Intelligence Report on Iran', current_situation='Trump ordered U.S. strikes on Iranian nuclear sites as part of its support for Israel, then days later announced a ceasefire agreement between the countries.', macro_enviroment='The situation involves U.S. military actions and diplomatic agreements in the Middle East, affecting geopolitical stability.', geopolitical='The geopolitical landscape is tense due to military actions and subsequent diplomatic agreements between the U.S. and Iran.', sentiment='The sentiment is likely mixed due to the aggressive military actions followed by a ceasefire agreement.', risk_management='High risk due to geopolitical tensions and military involvement.', Recommendation='Monitor the situation closely for further developments and potential impacts on related assets.')],
 [trading_recomendation(asset_name='Nvidia', current_situation='Nvidia shares are heading for a record close as investors gain confidence.', macro_e

In [25]:
content = []
for doc in retr_docs:
    content.append(doc.page_content)


content = ' '.join(content)
content

'Founded in London in 2004, Amplified became quickly established as the leading music-inspired lifestyle brand, specialising in artwork for retro and classic rock bands. Amplified Clothing went on to dominate the market.Musicians, celebrities, music fans and fashion enthusiasts across the world adopted Amplified as a firm favourite. Over a decade later and the brand continues to explore a wider world of music from genres and sub-genres that have never stopped influencing our lives. Music is the absolute inspiration for everything we do. Amplified makes some noise. nan nan A.B.C.L. born by passion and vision shared by Antonio and Mattia are the stitchings that bind two different worlds and lifestyles into one brand. Part residing in Tokyo and the other in Venice, we have the desire to bring the best of what’s Japanese and seamlessly combine that with international design. A.B.C. represents the start of things, the beginning of a masterpiece, and the L. derives from Antonio surname. We a

In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

pt = ChatPromptTemplate.from_template("""
        Answer the user query: {query} given the context: {context}.  
        Do not create yourself the answer, just use the context provided to answer the user query.       
"""
    )

parser = StrOutputParser()
chain = pt | llm | parser

chain.invoke({
    'context': content, 
    'query': query
})

'The context provided does not contain specific information about events or occurrences in London. It mentions Amplified Clothing, a music-inspired lifestyle brand founded in London in 2004, and its influence in the fashion and music world. Additionally, it references Camden Market in London as a source of inspiration for the Anerkjendt brand. However, it does not detail any specific events or happenings in London.'

In [33]:
k = 5
def rag(query, k, vs):
    retr_docs = vs.similarity_search(query, k)

    content = []
    for doc in retr_docs:
        content.append(doc.page_content)
    content = ' '.join(content)

    pt = ChatPromptTemplate.from_template("""
        Answer the user query: {query} given the context: {context}.  
        Do not create yourself the answer, just use the context provided to answer the user query.       
    """
        )

    parser = StrOutputParser()
    chain = pt | llm | parser

    response = chain.invoke({
        'context': content, 
        'query': query
    })
    
    return response



In [34]:
rag('what happened in london', k, vs)

'The context provided does not contain specific information about events or occurrences in London. It mentions Amplified Clothing, a music-inspired lifestyle brand founded in London in 2004, and its influence in the fashion and music world. Additionally, it references Camden Market in London as a source of inspiration for the Anerkjendt brand. However, it does not detail any specific events or happenings in London.'

In [35]:
rag('tell me something about Made in Italy', k, vs)

'"Made in Italy" represents a symbol of excellence and quality in craftsmanship, particularly in the fashion and luxury goods sectors. The context highlights several Italian companies that embody this tradition. Antonio Maurizi and A.Testoni are renowned for their artisanal shoemaking, with a focus on innovation and design, using 100% Italian components sourced for quality and environmental impact. A.Testoni, established in Bologna, has grown into an international luxury brand known for its unique combination of heritage and modern design techniques. Similarly, Alea, rooted in Romagna, has been a leader in high-quality shirt production since the 1950s, maintaining a balance between tradition and modernity. Alberto del Biondi S.p.A., founded in Padua, is a multidisciplinary company that combines design, innovation, and engineering, creating footwear, bags, and accessories for top international brands. These companies exemplify the "Made in Italy" ethos through their commitment to qualit

In [37]:
print(rag('which brand use innovation', k, vs))

The brands that use innovation according to the context provided are:

1. A.Testoni - The company has grown into an international luxury brand through constant innovation and design research, particularly in the use of leather, materials, and design.

2. AlphaTauri - This brand offers innovative fashion technologies integrated with high-quality and uniquely designed styles, combining textile innovations, purposeful design, and premium materials.


In [38]:
print(rag('which brand is a family business', k, vs))

The brand that is a family business given the context is Nipal, founded by Goliardo and Franco Lazzeri.


In [40]:
print(rag('are there any brands based on Naples', k, vs))

Yes, based on the context provided, Nipal is a company based on Naples.


In [41]:
print(rag('are there any brands specialized in jackets?', k, vs))

Yes, there are brands specialized in jackets mentioned in the context. Nipal designs and makes leather and fabric jackets for men and women. Additionally, Arctic Explorer presents different pieces of high-quality winter outerwear, including down jackets.


In [42]:
print(rag('are there any brands US based?', k, vs))

Yes, Allen Edmonds is a U.S.-based brand.


In [43]:
print(rag('list all brands that create products based on artisanal spirit', k, vs))

Based on the context provided, the brands that create products based on an artisanal spirit are:

1. Andrea Greco
2. Alessandro Gherardi
3. Antonio Maurizi
4. Nipal (including the brands Fontanelli, AFG 1972, Stilnology, and Reali26)
5. A.B.C.L.


In [44]:
print(rag('which products alessandro gherardi create?', k, vs))

Alessandro Gherardi creates belts, leather jewelry, small leather goods, bags, placemats, pillows, and carpets.


In [46]:
print(rag('what is the must to have from alessandro gherardi', k, vs))

The must-have from Alessandro Gherardi, given the context, would be items from the Alex Ingh collection. This collection is 100% Made in Italy and features high-range innovative tailoring with the finest Italian fabrics and original printings with exclusive details. It appeals to a younger audience with a sense of classical and traditional themes, offering a wardrobe that interprets the needs and personality of its user.
